### Making the dataset file structure compatible with the SDG Engine's

Convert the Homebrew dataset to an easily loadable format for Hugging Face's datasets library.

In [2]:
import os
from typing import List, Dict
import shutil
import json
from PIL import Image

dataset_jsonl: List[Dict] = []
dataset_path = "/Users/federico/Documents/personal/projects/sdg-engine-files/homebrew-scenes/val_primesense/000003"
target_dataset_path = "/Users/federico/Documents/personal/code/sdg-engine/tutorials/real-bop-homebrew-scene-3/val"

# if the target dataset path does not exist, create it
if not os.path.exists(target_dataset_path):
    os.makedirs(target_dataset_path)

# Path to the rgb images
rgb_images = os.listdir(os.path.join(dataset_path, "rgb"))

# load the scene_gt_info.json file
scene_gt_info = json.load(
    open(os.path.join(dataset_path, "scene_gt_info.json"))
)

# Iterate over the rgb images
for image in rgb_images:
    annotation_dict = {}
    # Copy the image to the target dataset path
    img_new_path = os.path.join(target_dataset_path, image)
    shutil.copy(
        os.path.join(dataset_path, "rgb", image),
        img_new_path,
    )
    # Collect the object's bounding box from the scene_gt_info.json file
    image_id = image.split(".")[0].lstrip("0") or "0"
    if len(scene_gt_info[image_id]) == 1:
        image_bbox = [scene_gt_info[image_id][0]["bbox_obj"]]
    else:
        image_bbox = [annotation["bbox_obj"] for annotation in scene_gt_info[image_id]]

    # Get the image dimensions  
    image_path = os.path.join(dataset_path, "rgb", image)
    img = Image.open(image_path)
    width, height = img.size

    # Populate the annotation dictionary
    annotation_dict["image_id"] = image_id
    annotation_dict["width"] = width # TODO: remove hardcoded width and height
    annotation_dict["height"] = height # TODO: remove hardcoded width and height
    annotation_dict["file_name"] = image
    annotation_dict["objects"] = {
        "bbox": image_bbox,
        "bbox_ids": [int(image_id)*1000+i for i in range(len(image_bbox))],
        "areas": [image_bbox[i][2]*image_bbox[i][3] for i in range(len(image_bbox))],
        "categories": [i for i in range(len(image_bbox))],
    }

    dataset_jsonl.append(annotation_dict)

# Finally, save the dataset_jsonl to a file
with open(f"{target_dataset_path}/metadata.jsonl", "w") as f:
    for annotation in dataset_jsonl:
        f.write(json.dumps(annotation) + "\n")

Now load them into Hugging Face datasets.

In [ ]:
from datasets import load_dataset

DATASET_PATH = "/Users/federico/Documents/personal/code/sdg-engine/tutorials/real-bop-homebrew-scene-3"
dataset = load_dataset("imagefolder", data_dir=DATASET_PATH)
print(f"Dataset loaded: \n{dataset}")

dataset.push_to_hub(
    "federicoarenas-ai/real-bop-homebrew-scene-3",
    token="",
    private=True
)

Dataset loaded: 
DatasetDict({
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'objects'],
        num_rows: 340
    })
})


Let's push this validation dataset to HuggingFace.

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")

dataset.push_to_hub(
    "federicoarenas-ai/real-bop-homebrew-scene-3",
    token="",
    private=True
)

Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.37s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/federicoarenas-ai/real-bop-homebrew-scene-3/commit/4361f93cfcb42d2a77e3b54dba1d7a88621b3444', commit_message='Upload dataset', commit_description='', oid='4361f93cfcb42d2a77e3b54dba1d7a88621b3444', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/federicoarenas-ai/real-bop-homebrew-scene-3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='federicoarenas-ai/real-bop-homebrew-scene-3'), pr_revision=None, pr_num=None)

### Push synthetic dataset to HF

In [ ]:
from datasets import load_dataset

DATASET_PATH = "/Users/federico/Documents/personal/code/sdg-engine/synthetic-bop-homebrew-scene-3-large"
dataset_render = load_dataset("imagefolder", data_dir=DATASET_PATH)
print(f"Dataset loaded: \n{dataset_render}")


Generating train split: 2197 examples [00:00, 15207.54 examples/s]


Dataset loaded: 
DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'objects'],
        num_rows: 2197
    })
})


Finally, lets push them to the Hugging Face Hub.

In [ ]:
dataset_render.push_to_hub(
    "federicoarenas-ai/synthetic-bop-homebrew-scene-3-medium",
    token="",
)

Uploading the dataset shards: 100%|██████████| 2/2 [00:33<00:00, 16.99s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/federicoarenas-ai/synthetic-bop-homebrew-scene-3-medium/commit/313c84606873777857a1b5b84c243b129a457bf7', commit_message='Upload dataset', commit_description='', oid='313c84606873777857a1b5b84c243b129a457bf7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/federicoarenas-ai/synthetic-bop-homebrew-scene-3-medium', endpoint='https://huggingface.co', repo_type='dataset', repo_id='federicoarenas-ai/synthetic-bop-homebrew-scene-3-medium'), pr_revision=None, pr_num=None)